In [2]:
import re
import pandas as pd
import numpy as np
import spacy
import random
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [19]:
def normalize_text(text: str) -> str:
    if pd.isna(text): return ""
    text = str(text)
    text = re.sub(r'[‘’`]', "'", text)
    text = re.sub(r'[“”]', '"', text)
    text = re.sub(r'[—–]', '--', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s+([.,;:!?])', r'\1', text)
    return text.strip()

class EntityExtractor:
    def __init__(self, model_name="en_core_web_trf"):
        print(f"Loading NER model ({model_name})...")
        self.nlp = spacy.load(model_name)
        
    def extract(self, df: pd.DataFrame) -> pd.DataFrame:
        print("Extracting Named Entities...")
        records = []
        for i, doc in enumerate(self.nlp.pipe(df['text'], batch_size=16)):
            target_labels = {'PERSON', 'GPE', 'LOC', 'DATE', 'TIME', 'FAC', 'ORG'}
            entities = {ent.text for ent in doc.ents if ent.label_ in target_labels}
            records.append({'page': df.iloc[i]['page'], 'entities': list(entities)})
        return pd.DataFrame(records)

class StylometricAnalyzer:
    def __init__(self, embed_model='all-MiniLM-L6-v2'):
        print(f"Loading Stylometry Bi-Encoder ({embed_model})...")
        self.embed_model = SentenceTransformer(embed_model)
        
    def extract_features(self, df: pd.DataFrame) -> np.ndarray:
        print("Extracting Stylometric Features...")
        texts = df['text'].tolist()
        embeddings = self.embed_model.encode(texts, show_progress_bar=False)
        
        style_feats = []
        for text in texts:
            words = re.findall(r'\b\w+\b', text.lower())
            sentences = [s for s in re.split(r'[.!?]+', text) if s.strip()]
            num_words = len(words) or 1
            
            avg_len = num_words / (len(sentences) or 1)
            ttr = len(set(words)) / num_words
            punct_freq = len(re.findall(r'[;,—\-]', text)) / num_words
            style_feats.append([avg_len, ttr, punct_freq])
            
        scaler = StandardScaler()
        scaled_style = scaler.fit_transform(np.array(style_feats))
        return np.hstack((embeddings, scaled_style))

def run_phase_1(df: pd.DataFrame, n_narrators=6) -> pd.DataFrame:
    print("\n--- STARTING PHASE I ---")
    df['text'] = df['text'].apply(normalize_text)
    
    extractor = EntityExtractor()
    df = df.merge(extractor.extract(df), on='page')
    
    analyzer = StylometricAnalyzer()
    features = analyzer.extract_features(df)
    
    print(f"Clustering into {n_narrators} narrators...")
    kmeans = KMeans(n_clusters=n_narrators, random_state=42, n_init=10)
    df['narrator_cluster'] = kmeans.fit_predict(features)
    
    print("Phase I Complete.")
    return df

def extract_edges(text: str, num_words=75):
    words = str(text).split()
    if len(words) < num_words * 2:
        mid = len(words) // 2
        return " ".join(words[:mid]), " ".join(words[mid:])
    return " ".join(words[:num_words]), " ".join(words[-num_words:])

In [20]:
def build_masked_transition_matrix(df: pd.DataFrame, ce_model_path: str) -> np.ndarray:
    """
    Builds the N x N probability matrix, but ONLY computes pairs that 
    Phase I determined belong to the same narrator cluster.
    """
    print("\n--- STARTING PHASE II ---")
    print(f"Loading Cross-Encoder from {ce_model_path}...")
    try:
        model = CrossEncoder(ce_model_path)
    except Exception as e:
        raise RuntimeError("Could not load Cross-Encoder. Did you train it?") from e

    N = len(df)
    texts = df['text'].tolist()
    clusters = df['narrator_cluster'].tolist()
    heads, tails = zip(*[extract_edges(t) for t in texts])
    
    # Initialize matrix with 0.0 (impossible transition)
    transition_matrix = np.zeros((N, N), dtype=float)
    
    total_inferences = 0
    print("Computing Intra-Cluster Inferences...")
    
    for i in range(N):
        inference_pairs = []
        target_indices = []
        
        for j in range(N):
            # THE BRIDGE: Only compute if i and j are in the same cluster
            if i != j and clusters[i] == clusters[j]:
                inference_pairs.append([tails[i], heads[j]])
                target_indices.append(j)
                total_inferences += 1
                
        if inference_pairs:
            # Predict only the valid pairs within this cluster
            probs = model.predict(inference_pairs)
            for idx, prob in zip(target_indices, probs):
                transition_matrix[i, idx] = prob
                
    # Prevent self-loops
    np.fill_diagonal(transition_matrix, -np.inf)
    
    print(f"Phase II Complete. Computed {total_inferences} inferences instead of {N*(N-1)}.")
    return transition_matrix

In [21]:
TEST_DATA_PATH = "./data/BookB_test.csv"
CROSS_ENCODER_PATH = "./models/fine_tuned_cross_encoder"

# Load Raw Data
raw_df = pd.read_csv(TEST_DATA_PATH)

# Run Phase I (Prep, NER, Stylometry, Clustering)
processed_df = run_phase_1(raw_df, n_narrators=6)

# Save Metadata snapshot
processed_df.to_csv("BookB_Phase1_Metadata.csv", index=False)

# Run Phase II (Masked Probability Matrix)
probability_matrix = build_masked_transition_matrix(processed_df, CROSS_ENCODER_PATH)

# Save the raw matrix so you don't have to recompute it during Phase III testing
np.save("BookB_Phase2_Matrix.npy", probability_matrix)
print("\nPipeline finished successfully! Matrix saved to 'BookB_Phase2_Matrix.npy'")


--- STARTING PHASE I ---
Loading NER model (en_core_web_trf)...


Extracting Named Entities...
Loading Stylometry Bi-Encoder (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting Stylometric Features...
Clustering into 6 narrators...
Phase I Complete.

--- STARTING PHASE II ---
Loading Cross-Encoder from ./models/fine_tuned_cross_encoder...


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Computing Intra-Cluster Inferences...
Phase II Complete. Computed 604 inferences instead of 3192.

Pipeline finished successfully! Matrix saved to 'BookB_Phase2_Matrix.npy'


In [6]:
import numpy as np
import pandas as pd
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

In [22]:
def solve_hamiltonian_path(probability_matrix: np.ndarray, time_limit_seconds: int = 15) -> list[int]:
    """
    Transforms the probability matrix into a cost matrix and uses OR-Tools 
    to solve the Shortest Hamiltonian Path Problem.
    """
    print("Transforming probabilities to routing costs...")
    N = probability_matrix.shape[0]
    
    # 1. Logarithmic Cost Transformation: C_{i,j} = -log(M_{i,j})
    # Clip probabilities to a tiny epsilon to prevent log(0) errors
    epsilon = 1e-9
    clipped_probs = np.clip(probability_matrix, epsilon, 1.0)
    cost_matrix = -np.log(clipped_probs)
    
    # Ensure the diagonal (self-transitions) is mathematically prohibited
    np.fill_diagonal(cost_matrix, 999999.0)
    
    # 2. Inject Virtual Dummy Node
    # Pad the matrix with an extra row and column of zeros.
    # The dummy node (index N) costs 0 to enter from any node and 0 to exit to any node.
    augmented_matrix = np.pad(cost_matrix, ((0, 1), (0, 1)), mode='constant', constant_values=0)
    
    # 3. Integer Conversion for OR-Tools
    # OR-Tools strictly requires integer costs. We multiply by a large scalar
    # to preserve the floating-point precision of our log costs.
    precision_scalar = 100000
    int_matrix = (augmented_matrix * precision_scalar).astype(int)

    # 4. Configure the TSP Solver
    print("Initializing OR-Tools Routing Solver...")
    # N+1 total nodes, 1 vehicle, Starting and Ending at the dummy node (index N)
    manager = pywrapcp.RoutingIndexManager(N + 1, 1, N)
    routing = pywrapcp.RoutingModel(manager)

    # Define the distance callback
    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return int(int_matrix[from_node][to_node])

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # 5. Define Search Parameters (Heuristics & Metaheuristics)
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    # First solution strategy: Find a fast, cheap initial path
    search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    # Metaheuristic: Use Guided Local Search (which includes k-opt swaps) to escape local minima
    search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    search_parameters.time_limit.seconds = time_limit_seconds

    print(f"Solving the Asymmetric TSP (Time limit: {time_limit_seconds}s)...")
    solution = routing.SolveWithParameters(search_parameters)

    if not solution:
        raise RuntimeError("The routing solver failed to find a valid solution.")

    # 6. Extract the Optimal Sequence
    print("Optimal route discovered. Extracting sequence...")
    index = routing.Start(0)
    optimal_route = []
    
    while not routing.IsEnd(index):
        node_index = manager.IndexToNode(index)
        # Append to our sequence, ignoring the virtual dummy node
        if node_index != N: 
            optimal_route.append(node_index)
        index = solution.Value(routing.NextVar(index))
    
    return optimal_route

In [26]:
print("--- Phase III: Global Resolution (ATSP) ---")
metadata_df = pd.read_csv("BookB_Phase1_Metadata.csv")
probability_matrix = np.load("BookB_Phase2_Matrix.npy")

# 2. Execute the Combinatorial Solver
# Adjust the time_limit_seconds based on matrix size. 15s is usually plenty for N=100.
optimal_indices = solve_hamiltonian_path(probability_matrix, time_limit_seconds=15)

# 3. Construct the Final Submission Formatting
print("Constructing final submission CSV...")

# Extract the original page identifiers using the optimized indices
shuffled_page_ids = metadata_df.iloc[optimal_indices]['page'].tolist()

# The evaluation requires a continuous 1-based index
submission_df = pd.DataFrame({
    'original_page': range(1, len(optimal_indices) + 1),
    'shuffled_page': shuffled_page_ids
})

# Save final output
output_filename = 'BookB_Final_Submission.csv'
submission_df.to_csv(output_filename, index=False)

print(f"Success! Final reconstructed narrative saved to '{output_filename}'.")
print("The algorithmic pipeline is complete.")

--- Phase III: Global Resolution (ATSP) ---
Transforming probabilities to routing costs...
Initializing OR-Tools Routing Solver...
Solving the Asymmetric TSP (Time limit: 15s)...


Optimal route discovered. Extracting sequence...
Constructing final submission CSV...
Success! Final reconstructed narrative saved to 'BookB_Final_Submission.csv'.
The algorithmic pipeline is complete.


In [27]:
import torch
import pandas as pd
import re
import itertools
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

def load_quantized_llm(base_model_id="meta-llama/Meta-Llama-3.1-8B", adapter_path=None):
    print(f"Loading {base_model_id} in 4-bit precision...")
    
    # Configure 4-bit quantization to save VRAM
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    tokenizer = AutoTokenizer.from_pretrained(base_model_id)
    model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        quantization_config=bnb_config,
        device_map="auto"
    )
    
    # If you have fine-tuned LoRA weights on Agatha Christie, load them here
    if adapter_path:
        print(f"Injecting PEFT LoRA adapters from {adapter_path}...")
        model = PeftModel.from_pretrained(model, adapter_path)
        
    return model, tokenizer

def score_narrative_coherence(model, tokenizer, pages_text: list[str]) -> int:
    """
    Prompts the LLM to evaluate the logical continuity of a sequence of pages
    and extract a strict penalty/coherence score from 1 to 10.
    """
    # Construct the prompt payload
    narrative_block = "\n\n--- NEXT PAGE ---\n\n".join(pages_text)
    
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert literary critic and archivist specializing in early 20th-century mystery novels. 
Your task is to read a sequence of fragmented pages and evaluate their chronological and logical coherence.
Look for broken dialogue, impossible actions (e.g., finding an object before it is lost), and abrupt scene changes.
Provide a Coherence Score from 1 (Completely Disjointed/Impossible) to 10 (Perfect Continuity).
Respond strictly with the integer score inside brackets, e.g., [8].<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Evaluate the following sequence:

{narrative_block}

Score:<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=10, 
            temperature=0.1, # Low temperature for deterministic scoring
            pad_token_id=tokenizer.eos_token_id
        )
        
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    
    # Parse the integer score using regex
    match = re.search(r'\[(\d+)\]', response)
    if match:
        return int(match.group(1))
    return 5 # Neutral fallback if the LLM hallucinates formatting

def refine_sequence_with_llm(model, tokenizer, metadata_df: pd.DataFrame, initial_route: list[int], window_size: int = 3) -> list[int]:
    """
    Applies a sliding window over the TSP route. If the LLM detects low coherence,
    it tests all local permutations (bubble-sort style) and selects the best one.
    """
    print(f"Beginning LLM Refinement Passes (Window Size: {window_size})...")
    refined_route = initial_route.copy()
    
    # Create a fast lookup dictionary for page texts
    text_lookup = dict(zip(metadata_df['page'], metadata_df['text']))
    
    # Slide a window across the narrative sequence
    i = 0
    while i <= len(refined_route) - window_size:
        current_window_indices = refined_route[i : i + window_size]
        current_texts = [text_lookup[metadata_df.iloc[idx]['page']] for idx in current_window_indices]
        
        # 1. Baseline Score
        baseline_score = score_narrative_coherence(model, tokenizer, current_texts)
        print(f"Window [{i}:{i+window_size}] Baseline Score: {baseline_score}/10")
        
        # 2. Trigger Threshold for Local Search
        # If the score is highly coherent (e.g., 8 or above), we trust the TSP solver and move on.
        if baseline_score >= 8:
            i += 1
            continue
            
        print(f"   Inconsistency detected at index {i}. Initiating local permutation search...")
        
        # Generate all possible permutations for this small window (e.g., 3! = 6 permutations)
        best_score = baseline_score
        best_permutation = current_window_indices
        
        for perm in itertools.permutations(current_window_indices):
            if list(perm) == current_window_indices:
                continue # Skip the baseline we already scored
                
            perm_texts = [text_lookup[metadata_df.iloc[idx]['page']] for idx in perm]
            score = score_narrative_coherence(model, tokenizer, perm_texts)
            
            if score > best_score:
                best_score = score
                best_permutation = list(perm)
                
        # 3. Apply the optimal swap if an improvement was found
        if best_permutation != current_window_indices:
            print(f"   >>> Permutation improvement found! New Score: {best_score}/10. Applying swap.")
            refined_route[i : i + window_size] = best_permutation
            # Rewind the window slightly to ensure the new swap flows into the previous pages
            i = max(0, i - 1) 
        else:
            i += 1 # Move forward if no permutation improved the logic
            
    print("LLM Refinement Complete.")
    return refined_route

In [ ]:
from huggingface_hub import login

In [28]:
print("--- Phase IV: Narrative Coherence Refinement ---")
        
# 1. Load Data and the Phase III TSP Output
metadata_df = pd.read_csv("BookB_Phase1_Metadata.csv")
tsp_submission_df = pd.read_csv("BookB_Final_Submission.csv")

# Convert the submission CSV back into a working list of indices
page_to_index = {page: idx for idx, page in enumerate(metadata_df['page'])}
initial_tsp_route = [page_to_index[page] for page in tsp_submission_df['shuffled_page']]

# 2. Load the Quantized LLM
# Replace 'path/to/lora' with your fine-tuned weights if applicable
llm_model, llm_tokenizer = load_quantized_llm(base_model_id="meta-llama/Meta-Llama-3.1-8B")

# 3. Execute the Refinement Layer
# A window size of 3 means it evaluates blocks of 3 pages and tests all 6 permutations.
refined_route_indices = refine_sequence_with_llm(
    llm_model, 
    llm_tokenizer, 
    metadata_df, 
    initial_tsp_route, 
    window_size=3
)

# 4. Save the Refined Final Output
shuffled_page_ids = metadata_df.iloc[refined_route_indices]['page'].tolist()

final_refined_df = pd.DataFrame({
    'original_page': range(1, len(refined_route_indices) + 1),
    'shuffled_page': shuffled_page_ids
})

final_refined_df.to_csv("BookB_Refined_Submission.csv", index=False)
print("Phase IV complete. Ultimate sequence saved to 'BookA_Refined_Submission.csv'.")

--- Phase IV: Narrative Coherence Refinement ---
Loading meta-llama/Meta-Llama-3.1-8B in 4-bit precision...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Beginning LLM Refinement Passes (Window Size: 3)...
Window [0:3] Baseline Score: 5/10
   Inconsistency detected at index 0. Initiating local permutation search...
Window [1:4] Baseline Score: 5/10
   Inconsistency detected at index 1. Initiating local permutation search...
Window [2:5] Baseline Score: 5/10
   Inconsistency detected at index 2. Initiating local permutation search...
Window [3:6] Baseline Score: 5/10
   Inconsistency detected at index 3. Initiating local permutation search...
Window [4:7] Baseline Score: 5/10
   Inconsistency detected at index 4. Initiating local permutation search...
Window [5:8] Baseline Score: 5/10
   Inconsistency detected at index 5. Initiating local permutation search...
Window [6:9] Baseline Score: 5/10
   Inconsistency detected at index 6. Initiating local permutation search...
Window [7:10] Baseline Score: 5/10
   Inconsistency detected at index 7. Initiating local permutation search...
Window [8:11] Baseline Score: 5/10
   Inconsistency detecte

In [31]:
print("--- Phase IV: Narrative Coherence Refinement ---")

# 1. Load Data and the Phase III TSP Output
metadata_df = pd.read_csv("./data/BookB_test.csv")
tsp_submission_df = pd.read_csv("./submissions/BookB_hyper_tuning.csv")

# Convert the submission CSV back into a working list of indices
page_to_index = {page: idx for idx, page in enumerate(metadata_df['page'])}
initial_tsp_route = [page_to_index[page] for page in tsp_submission_df['shuffled_page']]

# 2. Load the Quantized LLM
# Replace 'path/to/lora' with your fine-tuned weights if applicable
llm_model, llm_tokenizer = load_quantized_llm(base_model_id="meta-llama/Meta-Llama-3.1-8B")

# 3. Execute the Refinement Layer
# A window size of 3 means it evaluates blocks of 3 pages and tests all 6 permutations.
refined_route_indices = refine_sequence_with_llm(
    llm_model, 
    llm_tokenizer, 
    metadata_df, 
    initial_tsp_route, 
    window_size=3
)

# 4. Save the Refined Final Output
shuffled_page_ids = metadata_df.iloc[refined_route_indices]['page'].tolist()

final_refined_df = pd.DataFrame({
    'original_page': range(1, len(refined_route_indices) + 1),
    'shuffled_page': shuffled_page_ids
})

final_refined_df.to_csv("BookB_tsp_Submission.csv", index=False)
print("Phase IV complete. Ultimate sequence saved to 'BookA_Refined_Submission.csv'.")

--- Phase IV: Narrative Coherence Refinement ---
Loading meta-llama/Meta-Llama-3.1-8B in 4-bit precision...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Beginning LLM Refinement Passes (Window Size: 3)...
Window [0:3] Baseline Score: 5/10
   Inconsistency detected at index 0. Initiating local permutation search...
Window [1:4] Baseline Score: 5/10
   Inconsistency detected at index 1. Initiating local permutation search...
Window [2:5] Baseline Score: 5/10
   Inconsistency detected at index 2. Initiating local permutation search...
Window [3:6] Baseline Score: 5/10
   Inconsistency detected at index 3. Initiating local permutation search...
Window [4:7] Baseline Score: 5/10
   Inconsistency detected at index 4. Initiating local permutation search...
Window [5:8] Baseline Score: 5/10
   Inconsistency detected at index 5. Initiating local permutation search...
Window [6:9] Baseline Score: 5/10
   Inconsistency detected at index 6. Initiating local permutation search...
Window [7:10] Baseline Score: 5/10
   Inconsistency detected at index 7. Initiating local permutation search...
Window [8:11] Baseline Score: 5/10
   Inconsistency detecte